# Model training Iteration 4 
##### In the Keras TextVectorization class ‘output_mode’ is set at 'int'.

- Model 1: Model architecture based on example in https://www.geeksforgeeks.org/nlp/rnn-for-text-classifications-in-nlp/ with 20% recurrent dropout in bidirectional layer.
- Model 2: The same architecture as Model 1 with 20% dropout in both LSTM layers. A recurrent dropout in the first layer and a normal dropout in the second layer.
- Model 3: The same architecture as Model 1 with 20% recurrent dropout in the first LSTM layer and a normalization layer between both LSTM layers.

In [1]:
# Read necessary modules
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np 
import os
import tensorflow as tf
import re
import nltk
from nltk.corpus import stopwords
from collections import Counter

2025-11-20 14:51:24.841341: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763650285.011710      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763650285.059121      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
# Read dataset
train_df = pd.read_csv('/kaggle/input/nlp-getting-started/train.csv')
train_df

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [3]:
# Function to write model training results to an external location 
def write_away(name, model_results, path):

# Writing away the results 
    name = pd.DataFrame.from_dict(model_results)
    name.to_csv(path, index=False)
    return name

In [4]:
# Function to remove integers from selected columns
def remove_numbers(x):
    return re.sub(r'\d+', '', x)

In [5]:
# Remove integers from column text
train_df['text'] = train_df['text'].apply(remove_numbers)
train_df

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,", people receive #wildfires evacuation orders ...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1
...,...,...,...,...,...
7608,10869,NaN,NaN,Two giant cranes holding a bridge collapse int...,1
7609,10870,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1
7610,10871,NaN,NaN,M. [: UTC]?km S of Volcano Hawaii. http://t.co...,1
7611,10872,NaN,NaN,Police investigating after an e-bike collided ...,1


In [6]:
# function to remove stopwords from nltk corpus from strings
def basic_clean(x):
  """
  A simple function to clean up the data. All the words that
  are not designated as a stop word is then lemmatized after
  encoding and basic regex parsing are performed.
  """
  stops = set(stopwords.words('english'))
  words = re.sub(r'[^\w\s]', '', x).lower().split()
  return [word for word in words if word not in stops]

In [7]:
# apply function basic_clean to column text
train_df['text'] = train_df['text'].apply(basic_clean)

# apply lambda function to put column text in right format
train_df['text'] = train_df['text'].apply(lambda x: ', '.join(x))
train_df

,id,keyword,location,text,target
0,1,NaN,NaN,"deeds, reason, earthquake, may, allah, forgive...",1
1,4,NaN,NaN,"forest, fire, near, la, ronge, sask, canada",1
2,5,NaN,NaN,"residents, asked, shelter, place, notified, of...",1
3,6,NaN,NaN,"people, receive, wildfires, evacuation, orders...",1
4,7,NaN,NaN,"got, sent, photo, ruby, alaska, smoke, wildfir...",1
...,...,...,...,...,...
7608,10869,NaN,NaN,"two, giant, cranes, holding, bridge, collapse,...",1
7609,10870,NaN,NaN,"aria_ahrary, thetawniest, control, wild, fires...",1
7610,10871,NaN,NaN,"utckm, volcano, hawaii, httptcozdtoydebj",1
7611,10872,NaN,NaN,"police, investigating, ebike, collided, car, l...",1


In [8]:
# Put text and target variable in an array
x_train = np.array(train_df.text)
y_train = np.array(train_df.target)

In [9]:
# Tokenize the text 
encoder_int = tf.keras.layers.TextVectorization(
    max_tokens=2673,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    ngrams=1,
    output_mode='int',
    output_sequence_length=None,
    pad_to_max_tokens=False,
    vocabulary=None,
    idf_weights= None,
    sparse=False,
    ragged=False,
    encoding='utf-8',
    name=None
)
encoder_int

I0000 00:00:1763650364.223469      48 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1763650364.224161      48 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


<TextVectorization name=text_vectorization, built=False>

In [10]:
# Put train_df.text in tensorflow format
text_dataset = tf.data.Dataset.from_tensor_slices(train_df.text)
# Use encoder_tf to create the vocabulary 
encoder_int.adapt(text_dataset)

### Model 1 

In [11]:
 model_int = tf.keras.Sequential([
    tf.keras.layers.InputLayer(shape=(1,), dtype = tf.string),
    encoder_int,
    tf.keras.layers.Embedding(input_dim = len(encoder_int.get_vocabulary()), output_dim = 64, mask_zero=False),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences = True, recurrent_dropout = 0.2)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1)
])
model_int.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, None)           │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 64)       │       171,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, None, 128)      │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 282,561 (1.08 MB)

 Trainable params: 282,561 (1.08 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# training model 1
epochs = 20
batch_size = 150
model_int.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy", "auc"])

history_int = model_int.fit(x_train, y_train, batch_size= batch_size, epochs=epochs, validation_split=0.1)

In [ ]:
# Model 1: Writing away the results 
write_away('RNN_int_drop', history_int.history, 'RNN_int_drop.csv')

### Model 2

In [12]:
 model_int_I = tf.keras.Sequential([
    tf.keras.layers.InputLayer(shape=(1,), dtype = tf.string),
    encoder_int,
    tf.keras.layers.Embedding(input_dim = len(encoder_int.get_vocabulary()), output_dim = 64, mask_zero=False),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences = True, recurrent_dropout = 0.2)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, dropout = 0.2)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1)
])
model_int_I.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, None)           │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_1 (Embedding)         │ (None, None, 64)       │       171,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, None, 128)      │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 282,561 (1.08 MB)

 Trainable params: 282,561 (1.08 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# training model 2
epochs = 20
batch_size = 150
model_int_I.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy", "auc"])

history_int_I = model_int_I.fit(x_train, y_train, batch_size= batch_size, epochs=epochs, validation_split=0.1)

In [ ]:
# Model 2: Writing away the results 
write_away('RNN_int_I_drop', history_int_I.history, 'RNN_int_I_drop.csv')

### Model 3

In [13]:
 model_int_no = tf.keras.Sequential([
    tf.keras.layers.InputLayer(shape=(1,), dtype = tf.string),
    encoder_int,
    tf.keras.layers.Embedding(input_dim = len(encoder_int.get_vocabulary()), output_dim = 64, mask_zero=False),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences = True, recurrent_dropout = 0.2)),
    tf.keras.layers.LayerNormalization(), 
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)), 
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1)
])
model_int_no.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, None)           │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, None, 64)       │       171,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, None, 128)      │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization             │ (None, None, 128)      │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 282,817 (1.08 MB)

 Trainable params: 282,817 (1.08 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# training model 2
epochs = 20
batch_size = 150
model_int_no.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy", "auc"])

history_int_no = model_int_no.fit(x_train, y_train, batch_size= batch_size, epochs=epochs, validation_split=0.1)

In [ ]:
# Model 3: Writing away the results 
write_away('RNN_int_drno', history_int_no.history, 'RNN_int_drno.csv')